# F3 - Notebook 01: Algoritmos estructurados y recursivos

## Proyecto

**Análisis global de la industria fitness y gimnasios 2000-2026**

## Objetivo de este notebook

Este notebook contiene los algoritmos principales de la Fase 3 en formato **100% Jupyter Notebook**, sin archivos `.py`.

El propósito es demostrar:

- Programación estructurada mediante funciones, ciclos y condicionales.
- Programación recursiva mediante búsqueda de máximo.
- Comparación entre enfoques iterativos y recursivos.
- Coherencia entre datos de entrada, procesamiento y resultado esperado.
- Uso del dataset procesado proveniente de Fase 2.

## Resultado esperado

Al ejecutar este notebook se espera obtener:

- Dataset procesado cargado correctamente.
- Validación de columnas necesarias.
- Ranking de países según `memberships_per_100k`.
- Máximo calculado mediante tres enfoques.
- Promedio por región calculado manualmente y con Pandas.
- Filtros y búsquedas simples funcionando correctamente.

## 1. Relación con Fase 2

En Fase 2 se trabajó la limpieza, depuración, transformación y exportación del dataset procesado.

Por eso, en Fase 3 **no se vuelve a limpiar el dataset desde cero**. En cambio, se utiliza el archivo:

`data/processed/gym_data_processed.csv`

Este archivo debe contener variables derivadas como:

- `memberships_per_100k`
- `gyms_per_100k`
- `revenue_per_membership_usd`
- `periodo`

## Resultado esperado de esta sección

Se espera que el notebook cargue el dataset procesado y confirme que las columnas necesarias para los algoritmos existen.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

## 2. Configuración de rutas

Como este notebook está pensado para guardarse dentro de la carpeta `notebooks/`, la raíz del proyecto se obtiene con:

`Path("..").resolve()`

Desde esa raíz se construye la ruta hacia el dataset procesado:

`data/processed/gym_data_processed.csv`

## Resultado esperado

La celda debe imprimir:

- La ruta raíz del proyecto.
- La ruta del dataset procesado.
- Un mensaje de carga correcta.
- La cantidad de filas y columnas del dataset.

In [3]:
# Busca automáticamente la raíz del proyecto subiendo carpetas
def encontrar_raiz_proyecto(nombre_repo="gym-fitness-analytics"):
    ruta_actual = Path.cwd().resolve()

    for ruta in [ruta_actual] + list(ruta_actual.parents):
        if ruta.name == nombre_repo:
            return ruta

    raise FileNotFoundError(
        f"No se pudo encontrar la raíz del proyecto '{nombre_repo}' desde {ruta_actual}"
    )


PROJECT_ROOT = encontrar_raiz_proyecto()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "gym_data_processed.csv"

print("Directorio actual:", Path.cwd().resolve())
print("Raíz del proyecto:", PROJECT_ROOT)
print("Ruta del dataset procesado:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset procesado en la ruta: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset procesado cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

df.head()

Directorio actual: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics\src\fase3
Raíz del proyecto: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics
Ruta del dataset procesado: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics\data\processed\gym_data_processed.csv
Dataset procesado cargado correctamente.
Filas: 3564
Columnas: 27


,country,year,region,gym_memberships,fitness_participation_rate,total_health_club_revenue_usd,number_of_gyms,gym_penetration_rate,urban_population_percentage,obesity_rate,...,periodo,gym_memberships_norm,total_health_club_revenue_usd_norm,number_of_gyms_norm,gdp_per_capita_usd_norm,population_total_norm,average_membership_cost_usd_norm,memberships_per_100k_norm,gyms_per_100k_norm,revenue_per_membership_usd_norm
0,Angola,2000,Africa,95521.0,0.3873,5731259.0,204.0,0.0059,0.5051,0.0470,...,pre_covid,0.000909,0.000066,0.003230,0.003299,0.011038,0.0,0.012043,0.034128,0.000001
1,Angola,2001,Africa,103840.0,0.3939,6230372.0,222.0,0.0062,0.5172,0.0496,...,pre_covid,0.000990,0.000071,0.003519,0.003080,0.011419,0.0,0.013051,0.036453,0.000001
2,Angola,2002,Africa,121093.0,0.4003,7265583.0,249.0,0.0070,0.5289,0.0522,...,pre_covid,0.001158,0.000084,0.003953,0.006461,0.011819,0.0,0.015678,0.040381,0.000001
3,Angola,2003,Africa,142783.0,0.4065,8566966.0,281.0,0.0080,0.5400,0.0548,...,pre_covid,0.001369,0.000099,0.004467,0.007438,0.012243,0.0,0.018908,0.044930,0.000001
4,Angola,2004,Africa,179615.0,0.4124,10776918.0,325.0,0.0097,0.5504,0.0574,...,pre_covid,0.001728,0.000125,0.005174,0.009747,0.012696,0.0,0.024573,0.051321,0.000001


## 3. Validación de columnas necesarias

Antes de ejecutar algoritmos, se valida que el dataset contenga las columnas requeridas.

Esta validación es importante porque evita errores posteriores y demuestra coherencia entre:

- Datos de entrada.
- Procesamiento algorítmico.
- Resultados esperados.

## Resultado esperado

Si todas las columnas existen, se imprimirá:

`Todas las columnas necesarias para Fase 3 están presentes.`

Si falta alguna columna, se lanzará un error indicando cuáles faltan.

In [4]:
columnas_necesarias = [
    "country",
    "region",
    "year",
    "gym_penetration_rate",
    "memberships_per_100k",
    "gyms_per_100k",
    "revenue_per_membership_usd",
    "periodo"
]

faltantes = [col for col in columnas_necesarias if col not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias para Fase 3: {faltantes}")

print("Todas las columnas necesarias para Fase 3 están presentes.")

Todas las columnas necesarias para Fase 3 están presentes.


## 4. Preparación mínima del dataset

En Fase 3 no se realiza limpieza completa. Solo se seleccionan las columnas necesarias y se eliminan nulos puntuales en variables usadas por los algoritmos.

Esta decisión mantiene el foco en diseño algorítmico, no en preprocesamiento.

## Resultado esperado

Se espera obtener un nuevo DataFrame llamado `df_f3`, con las columnas necesarias y sin nulos en variables críticas para los algoritmos.

In [5]:
columnas_f3 = [
    "country",
    "region",
    "year",
    "gym_penetration_rate",
    "memberships_per_100k",
    "gyms_per_100k",
    "revenue_per_membership_usd",
    "periodo"
]

df_f3 = df[columnas_f3].copy()

df_f3 = df_f3.dropna(
    subset=[
        "gym_penetration_rate",
        "memberships_per_100k",
        "gyms_per_100k"
    ]
)

print("Dataset preparado para Fase 3.")
print("Filas disponibles:", df_f3.shape[0])
print("Columnas disponibles:", df_f3.shape[1])

df_f3.head()

Dataset preparado para Fase 3.
Filas disponibles: 3564
Columnas disponibles: 8


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
0,Angola,Africa,2000,0.0059,589.822616,1.259658,59.999990,pre_covid
1,Angola,Africa,2001,0.0062,620.043651,1.325594,59.999730,pre_covid
2,Angola,Africa,2002,0.0070,698.840625,1.437006,60.000025,pre_covid
3,Angola,Africa,2003,0.0080,795.727216,1.566008,59.999902,pre_covid
4,Angola,Africa,2004,0.0097,965.650082,1.747272,60.000100,pre_covid


## 5. Funciones auxiliares de validación

Estas funciones permiten validar que una o varias columnas existan antes de ejecutar cada algoritmo.

Esto mejora la robustez del código, ya que entrega errores claros cuando el dataset no cumple con la estructura esperada.

## Resultado esperado

Estas funciones no muestran resultados por sí solas. Se usarán internamente en los algoritmos.

In [6]:
def validar_columna(df, columna):
    """
    Valida que una columna exista dentro del DataFrame.

    Parámetros:
        df: DataFrame de entrada.
        columna: Nombre de la columna a validar.

    Resultado esperado:
        - No retorna nada si la columna existe.
        - Lanza ValueError si la columna no existe.
    """
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")


def validar_columnas(df, columnas):
    """
    Valida que una lista de columnas exista dentro del DataFrame.

    Parámetros:
        df: DataFrame de entrada.
        columnas: Lista de columnas requeridas.

    Resultado esperado:
        - No retorna nada si todas las columnas existen.
        - Lanza ValueError si falta una o más columnas.
    """
    faltantes = [col for col in columnas if col not in df.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")

# 6. Algoritmo 1: Ranking iterativo de países

## Qué hace este algoritmo

Calcula un ranking de países según una variable numérica. En este caso, se recomienda usar:

`memberships_per_100k`

El algoritmo:

1. Valida columnas requeridas.
2. Recorre el DataFrame fila por fila.
3. Extrae los valores válidos.
4. Construye una lista auxiliar.
5. Ordena la lista de mayor a menor.
6. Retorna los primeros `top_n` resultados.

## Complejidad

- Tiempo: `O(n log n)`, porque el ordenamiento domina el costo.
- Espacio: `O(n)`, porque se guarda una lista auxiliar.

## Resultado esperado

Una tabla con los 10 registros de mayor valor para `memberships_per_100k`.

In [7]:
def calcular_ranking_paises(df, columna, top_n=10):
    """
    Calcula un ranking de países según una columna numérica.

    Parámetros:
        df: DataFrame procesado.
        columna: Columna numérica usada para ordenar.
        top_n: Cantidad de registros a retornar.

    Resultado esperado:
        Lista de diccionarios con país, región, año y valor.
    """
    if top_n <= 0:
        raise ValueError("top_n debe ser mayor que cero.")

    columnas_requeridas = ["country", "region", "year", columna]
    validar_columnas(df, columnas_requeridas)

    resultados = []

    for _, fila in df.iterrows():
        valor = fila[columna]

        if pd.notna(valor):
            resultados.append({
                "country": fila["country"],
                "region": fila["region"],
                "year": int(fila["year"]),
                "valor": float(valor)
            })

    resultados_ordenados = sorted(
        resultados,
        key=lambda x: x["valor"],
        reverse=True
    )

    return resultados_ordenados[:top_n]

In [8]:
ranking_memberships = calcular_ranking_paises(
    df_f3,
    columna="memberships_per_100k",
    top_n=10
)

df_ranking_memberships = pd.DataFrame(ranking_memberships)

print("Top 10 países/registros según memberships_per_100k:")
df_ranking_memberships

Top 10 países/registros según memberships_per_100k:


,country,region,year,valor
0,United States of America,North America,2026,30221.918617
1,United States of America,North America,2025,30000.846077
2,Netherlands,Europe,2026,29194.416757
3,Norway,Europe,2023,28987.602546
4,Netherlands,Europe,2025,28980.856127
5,Netherlands,Europe,2024,28731.432516
6,Norway,Europe,2026,28597.186178
7,Norway,Europe,2025,28388.007851
8,Norway,Europe,2024,28143.691298
9,Australia,Oceania,2026,27699.213423


# 7. Algoritmo 2: Máximo iterativo

## Qué hace este algoritmo

Busca el valor máximo de una lista usando un ciclo `for`.

## Por qué es relevante

Es un ejemplo claro de programación estructurada:

- Usa una condición inicial.
- Usa una variable acumuladora.
- Recorre la lista una sola vez.
- Compara cada valor con el máximo actual.

## Complejidad

- Tiempo: `O(n)`
- Espacio: `O(1)`

## Resultado esperado

Un número correspondiente al valor máximo de `memberships_per_100k`.

In [9]:
def maximo_iterativo(valores):
    """
    Obtiene el valor máximo de una lista usando enfoque iterativo.

    Resultado esperado:
        Retorna el mayor valor de la lista o None si la lista está vacía.
    """
    if len(valores) == 0:
        return None

    maximo = valores[0]

    for valor in valores[1:]:
        if valor > maximo:
            maximo = valor

    return maximo

# 8. Algoritmo 3: Máximo recursivo con slicing

## Qué hace este algoritmo

Busca el máximo dividiendo la lista en dos mitades de forma recursiva.

## Casos del algoritmo

- Caso base 1: lista vacía.
- Caso base 2: lista con un solo elemento.
- Caso recursivo: dividir en dos mitades y comparar máximos parciales.

## Ventaja

Permite demostrar recursividad y divide and conquer.

## Limitación

Usa slicing, por ejemplo `valores[:mitad]`, lo que crea copias de listas y aumenta el uso de memoria.

## Complejidad

- Tiempo: `O(n)`
- Espacio: `O(n)` por las copias generadas.

## Resultado esperado

Debe entregar el mismo máximo que el algoritmo iterativo.

In [10]:
def maximo_recursivo(valores):
    """
    Obtiene el valor máximo de una lista usando recursividad con slicing.

    Resultado esperado:
        Retorna el mayor valor de la lista o None si la lista está vacía.
    """
    if len(valores) == 0:
        return None

    if len(valores) == 1:
        return valores[0]

    mitad = len(valores) // 2

    max_izq = maximo_recursivo(valores[:mitad])
    max_der = maximo_recursivo(valores[mitad:])

    if max_izq is None:
        return max_der

    if max_der is None:
        return max_izq

    return max(max_izq, max_der)

# 9. Algoritmo 4: Máximo recursivo optimizado

## Qué hace este algoritmo

También usa recursividad, pero evita crear sublistas.

En vez de usar slicing, utiliza índices:

- `inicio`
- `fin`

## Ventaja

Reduce el uso de memoria porque no crea copias parciales de la lista.

## Complejidad

- Tiempo: `O(n)`
- Espacio: `O(log n)` por la pila recursiva, si la división es balanceada.

## Resultado esperado

Debe entregar el mismo máximo que los otros dos enfoques.

In [11]:
def maximo_recursivo_optimizado(valores, inicio=0, fin=None):
    """
    Obtiene el valor máximo usando recursividad con índices.

    Resultado esperado:
        Retorna el mayor valor de la lista o None si el rango está vacío.
    """
    if fin is None:
        fin = len(valores)

    if inicio >= fin:
        return None

    if fin - inicio == 1:
        return valores[inicio]

    mitad = (inicio + fin) // 2

    max_izq = maximo_recursivo_optimizado(valores, inicio, mitad)
    max_der = maximo_recursivo_optimizado(valores, mitad, fin)

    if max_izq is None:
        return max_der

    if max_der is None:
        return max_izq

    return max(max_izq, max_der)

## 10. Ejecución comparativa de máximos

En esta celda se crea una lista desde la columna `memberships_per_100k` y se ejecutan los tres algoritmos.

## Resultado esperado

Los tres enfoques deben entregar exactamente el mismo resultado.

Si los resultados no coinciden, el `assert` generará un error.

In [12]:
valores_memberships = (
    df_f3["memberships_per_100k"]
    .dropna()
    .astype(float)
    .tolist()
)

print("Cantidad de valores disponibles:", len(valores_memberships))
print("Primeros 5 valores:", valores_memberships[:5])

max_iterativo = maximo_iterativo(valores_memberships)
max_recursivo = maximo_recursivo(valores_memberships)
max_recursivo_opt = maximo_recursivo_optimizado(valores_memberships)

print("Máximo iterativo:", max_iterativo)
print("Máximo recursivo con slicing:", max_recursivo)
print("Máximo recursivo optimizado:", max_recursivo_opt)

assert max_iterativo == max_recursivo == max_recursivo_opt

print("Validación correcta: las tres implementaciones retornan el mismo resultado.")

Cantidad de valores disponibles: 3564
Primeros 5 valores: [589.8226160396852, 620.0436514552157, 698.8406250593342, 795.7272163084206, 965.650082258882]
Máximo iterativo: 30221.9186167546
Máximo recursivo con slicing: 30221.9186167546
Máximo recursivo optimizado: 30221.9186167546
Validación correcta: las tres implementaciones retornan el mismo resultado.


# 11. Algoritmo 5: Promedio manual por región

## Qué hace este algoritmo

Calcula el promedio de una variable numérica agrupada por región sin usar `groupby`.

Usa:

- Un diccionario para acumular sumas.
- Un diccionario para contar registros.
- Un ciclo para recorrer el DataFrame.
- Un ciclo final para calcular promedios.

## Complejidad

- Tiempo: `O(n)`
- Espacio: `O(k)`, donde `k` es el número de regiones.

## Resultado esperado

Un diccionario donde cada región tiene su promedio asociado.

In [13]:
def promedio_manual_por_region(df, columna):
    """
    Calcula el promedio de una columna por región usando ciclos.

    Resultado esperado:
        Diccionario con región como clave y promedio como valor.
    """
    validar_columnas(df, ["region", columna])

    acumuladores = {}
    conteos = {}

    for _, fila in df.iterrows():
        region = fila["region"]
        valor = fila[columna]

        if pd.notna(region) and pd.notna(valor):
            region = str(region)

            if region not in acumuladores:
                acumuladores[region] = 0.0
                conteos[region] = 0

            acumuladores[region] += float(valor)
            conteos[region] += 1

    promedios = {}

    for region in acumuladores:
        if conteos[region] > 0:
            promedios[region] = acumuladores[region] / conteos[region]

    return promedios

# 12. Algoritmo 6: Promedio por región usando Pandas

## Qué hace este algoritmo

Calcula el promedio por región usando:

`df.groupby("region")[columna].mean()`

## Por qué se compara con el manual

Permite analizar:

- Legibilidad.
- Eficiencia práctica.
- Diferencia entre implementación manual y herramientas optimizadas.

## Resultado esperado

Una Serie de Pandas con los promedios por región ordenados de mayor a menor.

In [14]:
def promedio_pandas_por_region(df, columna):
    """
    Calcula el promedio de una columna por región usando Pandas.

    Resultado esperado:
        Serie de Pandas con promedios por región.
    """
    validar_columnas(df, ["region", columna])

    return df.groupby("region")[columna].mean().sort_values(ascending=False)

In [15]:
promedio_manual = promedio_manual_por_region(
    df_f3,
    "gym_penetration_rate"
)

promedio_pandas = promedio_pandas_por_region(
    df_f3,
    "gym_penetration_rate"
)

print("Promedio manual por región:")
display(pd.Series(promedio_manual).sort_values(ascending=False))

print("Promedio con Pandas por región:")
display(promedio_pandas)

Promedio manual por región:


Europe           0.113839
Oceania          0.098022
North America    0.088760
South America    0.055014
Asia             0.050762
Africa           0.013010
dtype: float64

Promedio con Pandas por región:


region
Europe           0.113839
Oceania          0.098022
North America    0.088760
South America    0.055014
Asia             0.050762
Africa           0.013010
Name: gym_penetration_rate, dtype: float64

# 13. Funciones adicionales: búsqueda lineal y filtrado

Estas funciones permiten demostrar estructuras de control adicionales.

## Búsqueda lineal

Recorre los registros uno por uno hasta encontrar coincidencias con un país.

## Filtrado por periodo

Filtra registros según la columna `periodo`.

## Resultado esperado

- Un DataFrame filtrado por país.
- Un DataFrame filtrado por periodo.

In [16]:
def busqueda_lineal_pais(df, pais):
    """
    Busca registros de un país usando recorrido lineal.

    Resultado esperado:
        DataFrame con registros del país buscado.
    """
    validar_columna(df, "country")

    coincidencias = []

    for _, fila in df.iterrows():
        if str(fila["country"]).lower() == pais.lower():
            coincidencias.append(fila)

    if len(coincidencias) == 0:
        return pd.DataFrame(columns=df.columns)

    return pd.DataFrame(coincidencias)


def filtrar_por_periodo(df, periodo):
    """
    Filtra registros por periodo.

    Resultado esperado:
        DataFrame filtrado por periodo.
    """
    validar_columna(df, "periodo")

    periodos_validos = {"pre_covid", "covid", "post_covid"}

    if periodo not in periodos_validos:
        raise ValueError(
            f"Periodo inválido: {periodo}. Use uno de: {periodos_validos}"
        )

    return df[df["periodo"] == periodo].copy()

In [17]:
pais_ejemplo = df_f3["country"].iloc[0]
resultado_busqueda = busqueda_lineal_pais(df_f3, pais_ejemplo)

print("País usado como ejemplo:", pais_ejemplo)
print("Registros encontrados:", resultado_busqueda.shape[0])

resultado_busqueda.head()

País usado como ejemplo: Angola
Registros encontrados: 27


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
0,Angola,Africa,2000,0.0059,589.822616,1.259658,59.999990,pre_covid
1,Angola,Africa,2001,0.0062,620.043651,1.325594,59.999730,pre_covid
2,Angola,Africa,2002,0.0070,698.840625,1.437006,60.000025,pre_covid
3,Angola,Africa,2003,0.0080,795.727216,1.566008,59.999902,pre_covid
4,Angola,Africa,2004,0.0097,965.650082,1.747272,60.000100,pre_covid


In [18]:
df_post_covid = filtrar_por_periodo(df_f3, "post_covid")

print("Registros del periodo post_covid:", df_post_covid.shape[0])

df_post_covid.head()

Registros del periodo post_covid: 660


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
22,Angola,Africa,2022,0.0304,3042.110615,4.237404,60.719986,post_covid
23,Angola,Africa,2023,0.0295,2948.978972,4.359195,59.999977,post_covid
24,Angola,Africa,2024,0.0290,2895.851694,4.384223,60.000026,post_covid
25,Angola,Africa,2025,0.0292,2920.993007,4.423815,59.999990,post_covid
26,Angola,Africa,2026,0.0294,2942.518195,4.455489,59.999975,post_covid


## 14. Conclusión del notebook de algoritmos

Este notebook implementa los algoritmos principales de Fase 3.

Se observa que:

- Para recorridos simples, el enfoque iterativo es directo y eficiente.
- La recursividad es útil para demostrar divide and conquer.
- La recursividad con slicing puede aumentar el uso de memoria.
- La versión recursiva con índices es una mejora concreta.
- Pandas ofrece mayor legibilidad para operaciones tabulares agrupadas.

Estos algoritmos serán usados en el notebook de mediciones para analizar tiempo y memoria.